# Requête HTTP 

Un requête HTTP est une requête basée sur le protocole TCP, elle fait partie de la couche application de la couche OSI. Elle permet d'accéder aux données mise à disposition sur une adresse IP (ou url résolue par un DNS) et un port. 

Les deux ports les plus utilisés dans le web sont le 80 pour les sites en HTTP et le 443 pour les sites en HTTPS. HTTPS est une variable du protocole HTTP basé sur le protocole TLS.

Il existe de nombreux types de requêtes selon la convention `REST`: 
- GET
- POST
- PUT 
- DELETE
- UPDATE.

Dans notre cas, nous allons utiliser la plupart du temps des GET et potentiellement des POST. 
- Le GET permet comme son nom l'indique de récupérer des informations en fonction de certains paramètres. 
- Le POST nécessite un envoi de données pour récupérer des données. Le body du post est, la plupart du temps, envoyé sous la forme d'un objet JSON.

Ces requêtes encapsulent un certain nombre de paramètres qui permettent soient d'identifier une provenance et un utilisateur ou de réaliser différentes actions.

In [1]:
import requests

In [2]:
url = "https://www.esiee.fr/"
response = requests.get(url)
response.status_code

200

Il existe deux méthodes pour récupérer le contenu de la page :

- `response.text` qui permet de retourner le texte sous la forme d'une chaine de charactères.
- `response.content` qui permet de récupérer le contenu de la page sous la forme de bytes

In [3]:
type(response.content)

bytes

In [4]:
type(response.text)

str

Pour récupérer les 1000 premiers charactères de la page :

In [5]:
response.text[0:1000]

'<!DOCTYPE html>\n<html lang="fr-FR">\n<head>\n\n<meta charset="utf-8">\n<!-- \n\tThis website is powered by TYPO3 - inspiring people to share!\n\tTYPO3 is a free open source Content Management Framework initially created by Kasper Skaarhoj and licensed under GNU/GPL.\n\tTYPO3 is copyright 1998-2026 of Kasper Skaarhoj. Extensions are copyright of their respective owners.\n\tInformation and contribution at https://typo3.org/\n-->\n\n\n\n<title>ESIEE Paris, l&#039;école de l&#039;innovation technologique | ESIEE Paris</title>\n<meta name="generator" content="TYPO3 CMS" />\n<meta name="description" content="Rejoignez ESIEE Paris, grande école d&#039;ingénieur dans les domaines des transitions numérique, énergétique et environnementale. Classée parmi les meilleures écoles d&#039;ingénieur selon le classement de l&#039;Etudiant. Habilitée par la Commission des Titres d&#039;Ingénieur (CTI). Membre de la Conférence des Grandes Ecoles (CGE). " />\n<meta name="viewport" content="width=device-w

Pour récupérer les headers HTTP de la réponse :

In [6]:
response.headers

{'Date': 'Tue, 24 Nov 2026 15:22:13 GMT', 'Server': 'Apache', 'Content-Language': 'fr', 'Vary': 'Accept-Encoding', 'Content-Encoding': 'gzip', 'X-UA-Compatible': 'IE=edge', 'X-Content-Type-Options': 'nosniff', 'Content-Length': '16944', 'Content-Type': 'text/html; charset=utf-8', 'X-Varnish': '30281828 18847868', 'Age': '95', 'Via': '1.1 varnish (Varnish/7.1)', 'Accept-Ranges': 'bytes', 'Connection': 'keep-alive'}

On peut modifier les paramètres de la requête et/ou ses headers. On peut par exemple ajouter un UserAgent (identifiant de l'initiateur de la requête) et un timeout de 10 secondes :

In [7]:
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}
response = requests.get(url, headers=headers, timeout = 10)
response.content[0:1000]

b'<!DOCTYPE html>\n<html lang="fr-FR">\n<head>\n\n<meta charset="utf-8">\n<!-- \n\tThis website is powered by TYPO3 - inspiring people to share!\n\tTYPO3 is a free open source Content Management Framework initially created by Kasper Skaarhoj and licensed under GNU/GPL.\n\tTYPO3 is copyright 1998-2026 of Kasper Skaarhoj. Extensions are copyright of their respective owners.\n\tInformation and contribution at https://typo3.org/\n-->\n\n\n\n<title>ESIEE Paris, l&#039;\xc3\xa9cole de l&#039;innovation technologique | ESIEE Paris</title>\n<meta name="generator" content="TYPO3 CMS" />\n<meta name="description" content="Rejoignez ESIEE Paris, grande \xc3\xa9cole d&#039;ing\xc3\xa9nieur dans les domaines des transitions num\xc3\xa9rique, \xc3\xa9nerg\xc3\xa9tique et environnementale. Class\xc3\xa9e parmi les meilleures \xc3\xa9coles d&#039;ing\xc3\xa9nieur selon le classement de l&#039;Etudiant. Habilit\xc3\xa9e par la Commission des Titres d&#039;Ing\xc3\xa9nieur (CTI). Membre de la Conf\xc3\x

## Exercice

## Exercice 1

- Créer une classe Python permettant de faire des requêtes HTTP.
- Cette classe doit utiliser toujours le même UserAgent.
- Le TimeOut sera spécifié à chaque appelle avec une valeur par défaut.
- Un mécanisme de retry sera mis en place de façon recursive.

In [8]:
import requests


class HTTPClient:

    def __init__(self):
        self.headers = {
            "User-Agent": "DataEngineering-ESIEE/1.0"
        }

    def get(self, url, timeout=10, retries=3):
        try:
            response = requests.get(
                url,
                headers=self.headers,
                timeout=timeout
            )
            response.raise_for_status()
            return response

        except requests.RequestException:
            if retries > 0:
                return self.get(url, timeout, retries - 1)
            raise

In [9]:
# Test de la classe HTTPClient

client = HTTPClient()

response = client.get("https://www.esiee.fr/", timeout=10)

print("Code HTTP :", response.status_code)

Code HTTP : 200


In [10]:
# Test du mécanisme de retry

try:
    client.get(
        "https://example.invalid",
        timeout=2,
        retries=3
    )
except requests.RequestException as e:
    print("Échec après toutes les tentatives :", e)

Échec après toutes les tentatives : HTTPSConnectionPool(host='example.invalid', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x10a84b750>: Failed to resolve 'example.invalid' ([Errno 8] nodename nor servname provided, or not known)"))


## Exercice 2

- Faire une fonction permettant de supprimer tous les espaces supperflus d'une string
- Faire une fonction qui prend une string html et renvois une string intelligible (enlever les caractères spéciaux,
- Récupérer le domaine en fonction d'un url

In [11]:
def clean_spaces(text):
    return " ".join(text.split())


# Test de la fonction
text = "  Bonjour    je suis   étudiante \n en Data Engineering.  "

print(clean_spaces(text))

Bonjour je suis étudiante en Data Engineering.


In [12]:
from bs4 import BeautifulSoup

def clean_html(html):
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(separator=" ")
    return clean_spaces(text)


# Test de la fonction
html = "<h1>Bonjour</h1><p>Bienvenue en Data Engineering !</p>"

print(clean_html(html))

Bonjour Bienvenue en Data Engineering !


In [13]:
from urllib.parse import urlparse

def get_domain(url):
    return urlparse(url).hostname


# Test de la fonction
print(get_domain("https://www.esiee.fr/formations"))
print(get_domain("https://www.google.com/search?q=python"))

www.esiee.fr
www.google.com


# Exploitation du HTML  

Ici, il faut récupérer le code HTML d'un site web à partir d'une requête. Lorsque vous avez récupéré le texte d'un site il faut le parser. Pour cela, on utilise BeautifulSoup qui permet de transformer la structure HTML en objet Python. Cela permet de récupérer efficacement les données qui nous intéresse.

Pour les webmasters, le blocage le plus souvent mis en place et un blocage sur le User-Agent. Le User-Agent est un paramètre intégré dans la requête HTTP réalisé par le Navigateur pour envoyer au front des informations basiques :

- la version du Navigateur,
- la version de l'OS
- Le type de gestionnaire graphique (Gecko)
- le type de device utilisé

Exemple de User Agent :  

`Mozilla/5.0 (Windows NT 6.1; Win64; x64; rv:47.0) Gecko/20100101 Firefox/47.0`

Commençons à utiliser `BeautifulSoup`, il est normalement déjà installé, le cas échéant executez les lignes suivantes : 

In [14]:
import requests
from bs4 import BeautifulSoup

Pour transformer une requête (requests) en objet BeautifulSoup :

In [15]:
response = requests.get(url)
soup = BeautifulSoup(response.text)

Pour trouver tous les liens d'une page, on récupère la balise `a` qui permet de gérer les liens en HTML :

In [16]:
soup.find_all("a")[0:10]

[<a href="/#content">Aller au contenu</a>,
 <a href="/#menu">Aller au menu</a>,
 <a href="/plan-du-site/">Plan du site</a>,
 <a href="/actualites/journees-portes-ouvertes-2026-2027" target="_blank" title="Ouvre une nouvelle fenêtre">Portes Ouvertes ESIEE Paris - Rendez-vous samedi 12 décembre de 13h à 18h </a>,
 <a href="/"><img alt="ESIEE PARIS" class="a42-ac-replace-img" src="/typo3conf/ext/esiee_sitepackage/Resources/Public/imgs/svg/logo-esiee.svg"/></a>,
 <a href="/brochures-1">Brochures</a>,
 <a href="/informations/etudiantes-et-etudiants">Espace élèves</a>,
 <a href="/" hreflang="fr-FR" title="Français">
 <span>Fr</span>
 </a>,
 <a href="/en/" hreflang="en-US" title="English">
 <span>En</span>
 </a>,
 <a href="/candidater-1">Candidater</a>]

On peut aussi préciser la classe HTML qu'on veut récupérer :

```python
soup.find_all(class_="<CLASS_NAME>")[0:10]
```

Ici par exemple: 

In [17]:
soup.find_all(class_="toggler")[0:5]

[<button aria-controls="searchbox-header-form" aria-expanded="false" class="toggler">
 <i class="fa-solid fa-magnifying-glass"></i>
 <i class="fa-solid fa-xmark"></i>
 <span class="sr-only">
 <span class="display">Afficher</span><span class="hide">Masquer</span> la recherche
 		</span>
 </button>,
 <button aria-controls="submenu-40" aria-expanded="false" class="toggler"><span class="sr-only"><span class="display">Afficher</span><span class="hide">Masquer</span> le sous menu : </span>L'école</button>,
 <button aria-controls="submenu-563" aria-expanded="false" class="toggler"><span class="sr-only"><span class="display">Afficher</span><span class="hide">Masquer</span> le sous menu : </span>Gouvernance et conseils</button>,
 <button aria-controls="submenu-65" aria-expanded="false" class="toggler"><span class="sr-only"><span class="display">Afficher</span><span class="hide">Masquer</span> le sous menu : </span>Départements d'enseignements et de recherche</button>,
 <button aria-controls="su

Pour récupérer le text sans les balises HTML :

In [18]:
soup.text[0:1000]

"\n\n\n\n\nESIEE Paris, l'école de l'innovation technologique | ESIEE Paris\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAller au contenu\nAller au menu\nPlan du site\n\n\n\n\n\n\n\nPortes Ouvertes ESIEE Paris - Rendez-vous samedi 12 décembre de 13h à 18h \n\n\n\n\n\nMasquer l'alerte\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nBrochuresEspace élèves\n\n\n\nFr\n\n\n\n\nEn\n\n\n\n\n\n\n\n\n\nAfficherMasquer la recherche\r\n\t\t\n\n\n\nSaisissez votre recherche\xa0:\n\nLancer la recherche\n\n\n\nCandidater\n\nAfficherMasquer le menu\n\n\n\n\n\nRetour au menu principalAfficherMasquer le sous menu\xa0: L'écolePourquoi choisir ESIEE Paris ?AfficherMasquer le sous menu\xa0: Gouvernance et conseilsGouvernance et conseilsConseil scientifiqueAfficherMasquer le sous menu\xa0: Départements d'enseignements et de rechercheInformatique et télécommunicationsIngénierie des systèmes cyberphysiquesIngénierie industrielleSanté, énergie et environnement durableManagement, sciences humaines et languesCorps professor

## Exercice
### Exercice 3

Améliorer la classe développé précédemment.

- Ajouter une méthode pour récupérer l'objet soup d'un url
- Récupérer une liste de User Agent et effectuer une rotation aléatoire sur celui à utiliser
- Utiliser cette classe pour parser une page HTML et récupérer : le titre, tous les H1 (si ils existent), les liens vers les images, les liens sortants vers d'autres sites, et le texte principal.

In [19]:
import random
import requests
from bs4 import BeautifulSoup


class HTTPClient:

    def __init__(self):
        self.user_agents = [
            "DataEngineering-ESIEE/1.0",
            "DataEngineering-ESIEE/2.0",
            "DataEngineering-ESIEE/3.0"
        ]

    def get(self, url, timeout=10, retries=3):
        headers = {
            "User-Agent": random.choice(self.user_agents)
        }

        try:
            response = requests.get(
                url,
                headers=headers,
                timeout=timeout
            )
            response.raise_for_status()
            return response

        except requests.RequestException:
            if retries > 0:
                return self.get(url, timeout, retries - 1)
            raise

    def get_soup(self, url, timeout=10):
        response = self.get(url, timeout)
        return BeautifulSoup(response.text, "html.parser")

In [20]:
client = HTTPClient()

url = "https://www.esiee.fr/"
soup = client.get_soup(url)

print("Type de l'objet :", type(soup))
print("Titre de la page :", soup.title.get_text())

Type de l'objet : <class 'bs4.BeautifulSoup'>
Titre de la page : ESIEE Paris, l'école de l'innovation technologique | ESIEE Paris


In [21]:
# Récupération de tous les titres H1
h1_titles = [
    clean_spaces(h1.get_text())
    for h1 in soup.find_all("h1")
]

print("Titres H1 :", h1_titles)

Titres H1 : ['']


In [22]:
from urllib.parse import urljoin

# Récupération des liens vers les images
images = [
    urljoin(url, img["src"])
    for img in soup.find_all("img", src=True)
]

print("Nombre d'images :", len(images))
print("Les 5 premières images :")

for image in images[:5]:
    print(image)

Nombre d'images : 90
Les 5 premières images :
https://www.esiee.fr/typo3conf/ext/esiee_sitepackage/Resources/Public/imgs/svg/logo-esiee.svg
data:image/gif;base64,R0lGODlhAQABAIAAAP///wAAACH5BAEAAAAALAAAAAABAAEAAAICRAEAOw==
https://www.esiee.fr/fileadmin/user_upload/Fichiers/image-home/HomeESIEEParis2026-1920x1080.webp
data:image/gif;base64,R0lGODlhAQABAIAAAP///wAAACH5BAEAAAAALAAAAAABAAEAAAICRAEAOw==
https://www.esiee.fr/fileadmin/_processed_/9/5/csm_actu-jpo-1344x840_6889669208.jpg


In [23]:
# Récupération des liens externes
external_links = []

for link in soup.find_all("a", href=True):
    full_url = urljoin(url, link["href"])

    if full_url.startswith(("http://", "https://")):
        if get_domain(full_url) != get_domain(url):
            external_links.append(full_url)

# Suppression des doublons
external_links = list(dict.fromkeys(external_links))

print("Nombre de liens externes :", len(external_links))

for link in external_links[:10]:
    print(link)

Nombre de liens externes : 12
http://esiee.jobteaser.com/fr/recruiter_account/job_offers
https://www.cge.asso.fr/
https://www.cti-commission.fr/
https://www.cti-commission.fr/la-cti/demarche-qualite/systeme-qe/eur-ace
https://www.enseignementsup-recherche.gouv.fr/fr/labels-des-formations-controlees-par-l-etat-46088
https://www.facebook.com/esieeparis
https://www.instagram.com/esieeparis/
https://www.linkedin.com/edu/school?id=20041&trk=tyah&trkInfo=idx%3A4-1-4%2CtarId%3A1425998561386%2Ctas%3Aesiee+paris
https://bsky.app/profile/esiee.fr
http://www.youtube.com/esieeparis


In [24]:
# Récupération du contenu principal
main = soup.find("main")

if main is not None:
    main_text = clean_spaces(main.get_text(separator=" "))
else:
    main_text = clean_html(str(soup))

print("Nombre de caractères :", len(main_text))
print("Aperçu du texte principal :")
print(main_text[:1500])

Nombre de caractères : 11197
Aperçu du texte principal :
ESIEE Paris, l’école de l'innovation technologique ESIEE Paris, 1re école d’ingénieurs post-bac selon le dernier classement de l'Usine Nouvelle 2026. Classement L'Etudiant 2026 ESIEE Paris, n°3 des écoles d’ingénieurs post-bac d’Ile-de-France selon le Palmarès « L’Etudiant 2026 ». 8 mars 2026 Journée internationale des droits des femmes ESIEE Paris s'engage dans la féminisation des métiers d'ingénieurs. En 16 ans, ESIEE Paris est passée de 10 % à 22 % de femmes parmi ses élèves. Découvez 25 parcours de femmes ingénieures, diplômées ESIEE Paris Visite virtuelle : ESIEE Paris, comme si vous y étiez Grâce à nos deux visites virtuelles immersives à 360° , découvrez l’école dans ses deux bâtiments, Perrault et Copernic. SecurESIEE : un programme innovant pour accompagner la réussite des élèves ESIEE Paris propose un programme d’accompagnement des élèves de premier cycle soutenu par le Fonds Social Européen (FSE+) et la Région Ile-de-F

# Exploitation des appels d'API



Losque le front du site récupère des données sur une API gérée par le back, un appel d'API est réalisé. Cet appel est recensé dans les appels réseaux. Il est alors possible de re-jouer cet appel pour récupérer à nouveau les données. Il est très facile de récupérer ces appels dans l'onglet Network de la console développeur de Chrome ou FireFox. La console vous permet de copier le code CURL de la requête et vous pouvez ensuite la transformer en code Python depuis le site https://curl.trillworks.com/.

Souvent les APIs sont bloquées avec certains paramètres. L'API vérifie que dans les headers de la requête HTTP ces paramètres sont présents :
* un token généré à la volée avec des protocoles OAuth2 (ou moins développés).
* un referer provenant du site web (la source de la requête), très facile à falsifier.



## Exercice 
### Exercice 4

- Utiliser les informations développées plus haut pour récupérer les premiers résultats d'une recherche d'une requête
sur Google. 

Tips : 

- Ouvrir les outils de développements de Chrome ou Firefox
- Onglet Network
- Fouiller dans les requêtes pour voir à quoi ressemble un appel API Google
- Utilisez beautiful soup pour convertir le contenu de la request en objet et accéder aux balises

In [25]:
# Recherche Google
query = "Data Engineering"

google_url = "https://www.google.com/search"

response = requests.get(
    google_url,
    params={"q": query},
    timeout=10
)

print("URL :", response.url)
print("Code HTTP :", response.status_code)

URL : https://consent.google.com/ml?continue=https://www.google.com/search%3Fq%3DData%2BEngineering&gl=FR&m=0&pc=srp&uxe=bip&cm=2&hl=fr&src=1
Code HTTP : 200


In [26]:
# Transformation de la réponse HTML
soup_google = BeautifulSoup(response.text, "html.parser")

# Récupération des titres des résultats
results = soup_google.find_all("h3")

print("Nombre de résultats :", len(results))

for title in results[:10]:
    print(clean_spaces(title.get_text()))

Nombre de résultats : 0


In [29]:
# Extraction des titres et des liens
google_results = []

for h3 in soup_google.find_all("h3")[:10]:
    link = h3.find_parent("a")

    if link and link.get("href"):
        google_results.append({
            "title": clean_spaces(h3.get_text()),
            "url": urljoin(response.url, link["href"])
        })

# Affichage des résultats
for result in google_results:
    print(result)

In [32]:
import sys

print(sys.executable)
print(sys.version)

/usr/local/bin/python3
3.13.7 (v3.13.7:bcee1c32211, Aug 14 2025, 19:10:51) [Clang 16.0.0 (clang-1600.0.26.6)]


In [42]:
uv run python -c "import lxml; print(lxml.__version__)"

/usr/local/bin/python3: No module named uv
Note: you may need to restart the kernel to use updated packages.


In [44]:
import xml.etree.ElementTree as ET

# Récupération du flux RSS
rss_url = "https://www.lemonde.fr/pixels/rss_full.xml"

response_rss = client.get(rss_url)

# Analyse du contenu XML
root = ET.fromstring(response_rss.content)

articles = root.findall(".//item")

print("Nombre d'articles :", len(articles))

Nombre d'articles : 20


In [45]:
news = []

for article in articles[:5]:
    news.append({
        "title": article.findtext("title"),
        "url": article.findtext("link"),
        "category": "Technologies"
    })

for article in news:
    print(article)

{'title': 'Nvidia\xa0: Donald Trump aurait gagné jusqu’à 15\xa0millions de dollars grâce à ses actions après le retour de Jensen Huang, selon le «\xa0Financial Times\xa0»', 'url': 'https://www.lemonde.fr/international/article/2026/11/24/nvidia-donald-trump-aurait-gagne-jusqu-a-15-millions-de-dollars-grace-a-ses-actions-apres-le-retour-de-jensen-huang-selon-le-financial-times_6814756_3210.html', 'category': 'Technologies'}
{'title': 'Arnaque aux sentiments\xa0: la banque doit-elle rembourser la victime\xa0?', 'url': 'https://www.lemonde.fr/argent/article/2026/11/24/arnaque-aux-sentiments-la-banque-doit-elle-rembourser-la-victime_6814746_1657007.html', 'category': 'Technologies'}
{'title': 'Jensen Huang de retour à la tête de Nvidia, sous l’influence de Donald Trump', 'url': 'https://www.lemonde.fr/international/video/2026/11/24/jensen-huang-de-retour-a-la-tete-de-nvidia-sous-l-influence-de-donald-trump_6814742_3210.html', 'category': 'Technologies'}
{'title': 'Guadeloupe\xa0: le parquet

In [46]:
for article in news:
    article["domain"] = get_domain(article["url"])

    soup_article = client.get_soup(article["url"])

    article["page_title"] = (
        clean_spaces(soup_article.title.get_text())
        if soup_article.title else None
    )

    print("Domaine :", article["domain"])
    print("Titre de la page :", article["page_title"])
    print("-" * 50)

Domaine : www.lemonde.fr
Titre de la page : Client Challenge
--------------------------------------------------
Domaine : www.lemonde.fr
Titre de la page : Client Challenge
--------------------------------------------------
Domaine : www.lemonde.fr
Titre de la page : Client Challenge
--------------------------------------------------
Domaine : www.lemonde.fr
Titre de la page : Client Challenge
--------------------------------------------------
Domaine : www.lemonde.fr
Titre de la page : Client Challenge
--------------------------------------------------


In [47]:
for article in news:
    soup_article = client.get_soup(article["url"])

    main = soup_article.find("article")

    if main is None:
        main = soup_article.find("main")

    if main is not None:
        article["text"] = clean_spaces(
            main.get_text(separator=" ")
        )
    else:
        article["text"] = clean_html(str(soup_article))

    print("Titre :", article["title"])
    print("Nombre de caractères :", len(article["text"]))
    print("Aperçu :", article["text"][:300])
    print("-" * 50)

Titre : Nvidia : Donald Trump aurait gagné jusqu’à 15 millions de dollars grâce à ses actions après le retour de Jensen Huang, selon le « Financial Times »
Nombre de caractères : 303
Aperçu : Client Challenge JavaScript is disabled in your browser. Please enable JavaScript to proceed. A required part of this site couldn’t load. This may be due to a browser extension, network issues, or browser settings. Please check your connection, disable any ad blockers, or try using a different brows
--------------------------------------------------
Titre : Arnaque aux sentiments : la banque doit-elle rembourser la victime ?
Nombre de caractères : 303
Aperçu : Client Challenge JavaScript is disabled in your browser. Please enable JavaScript to proceed. A required part of this site couldn’t load. This may be due to a browser extension, network issues, or browser settings. Please check your connection, disable any ad blockers, or try using a different brows
------------------------------------------

# Exercice Final  

Exercice Final
Utilisez tout ce que vous avez appris pour récupérer des articles de News avec une catégorie. Il est souvent intéressant de partir des flux RSS pour commencer :

Les données doivent comprendre :
- Le texte important propre
- L'url
- Le domaine
- la catégorie
- Le titre de l'article
- Le titre de la page
- (Facultatif) : les images

Tips : 

- Taper le nom de votre média favoris + RSS (par exemple : https://www.lemonde.fr/rss/)
- Aller dans le DOM de la page 
- Trouver les catégories et les liens vers les articles

In [48]:
# Récupération des images de chaque article

for article in news:
    soup_article = client.get_soup(article["url"])

    article["images"] = list(dict.fromkeys(
        urljoin(article["url"], img["src"])
        for img in soup_article.find_all("img", src=True)
        if not img["src"].startswith("data:")
    ))

    print("Titre :", article["title"])
    print("Nombre d'images :", len(article["images"]))
    print("-" * 50)

Titre : Nvidia : Donald Trump aurait gagné jusqu’à 15 millions de dollars grâce à ses actions après le retour de Jensen Huang, selon le « Financial Times »
Nombre d'images : 1
--------------------------------------------------
Titre : Arnaque aux sentiments : la banque doit-elle rembourser la victime ?
Nombre d'images : 1
--------------------------------------------------
Titre : Jensen Huang de retour à la tête de Nvidia, sous l’influence de Donald Trump
Nombre d'images : 1
--------------------------------------------------
Titre : Guadeloupe : le parquet de Paris ouvre une enquête à la suite de la cyberattaque contre le CHU
Nombre d'images : 1
--------------------------------------------------
Titre : Jensen Huang réintégré chez Nvidia : ce qu’il faut en retenir
Nombre d'images : 1
--------------------------------------------------


In [49]:
import pandas as pd

# Création du DataFrame
df_news = pd.DataFrame(news)

# Organisation des colonnes
df_news = df_news[
    ["title", "page_title", "url", "domain",
     "category", "text", "images"]
]

df_news

,title,page_title,url,domain,category,text,images
0,Nvidia : Donald Trump aurait gagné jusqu’à 15 ...,Client Challenge,https://www.lemonde.fr/international/article/2...,www.lemonde.fr,Technologies,Client Challenge JavaScript is disabled in you...,[https://www.lemonde.fr/_fs-ch-1T1wmsGaOgGaSxc...
1,Arnaque aux sentiments : la banque doit-elle r...,Client Challenge,https://www.lemonde.fr/argent/article/2026/11/...,www.lemonde.fr,Technologies,Client Challenge JavaScript is disabled in you...,[https://www.lemonde.fr/_fs-ch-1T1wmsGaOgGaSxc...
2,"Jensen Huang de retour à la tête de Nvidia, so...",Client Challenge,https://www.lemonde.fr/international/video/202...,www.lemonde.fr,Technologies,Client Challenge JavaScript is disabled in you...,[https://www.lemonde.fr/_fs-ch-1T1wmsGaOgGaSxc...
3,Guadeloupe : le parquet de Paris ouvre une enq...,Client Challenge,https://www.lemonde.fr/pixels/article/2026/11/...,www.lemonde.fr,Technologies,Client Challenge JavaScript is disabled in you...,[https://www.lemonde.fr/_fs-ch-1T1wmsGaOgGaSxc...
4,Jensen Huang réintégré chez Nvidia : ce qu’il ...,Client Challenge,https://www.lemonde.fr/international/live/2026...,www.lemonde.fr,Technologies,Client Challenge JavaScript is disabled in you...,[https://www.lemonde.fr/_fs-ch-1T1wmsGaOgGaSxc...


In [50]:
# Vérification de la qualité des données

print("Nombre d'articles :", len(df_news))
print("Nombre de colonnes :", len(df_news.columns))

print("\nValeurs manquantes :")
print(df_news.isna().sum())

print("\nNombre d'URL dupliquées :")
print(df_news["url"].duplicated().sum())

print("\nNombre de textes vides :")
print(df_news["text"].fillna("").str.strip().eq("").sum())

Nombre d'articles : 5
Nombre de colonnes : 7

Valeurs manquantes :
title         0
page_title    0
url           0
domain        0
category      0
text          0
images        0
dtype: int64

Nombre d'URL dupliquées :
0

Nombre de textes vides :
0
